# Accessing Full-Text of CIQ Transcripts using R
### Author: WRDS (adapted for Mastercard, Visa, American Express)
### Last Updated: November 2024

The full text of each transcript is stored separately from the metadata due to its size. This notebook uses a **local** WRDS connection (Renviron: `WRDS_USERNAME`, `WRDS_PASSWORD`) and looks up company IDs for **Mastercard**, **Visa**, and **American Express**.

### Connect to WRDS
Uses `connect_wharton()` from the **wrds** package, which reads `WRDS_USERNAME` and `WRDS_PASSWORD` from `.Renviron`. You can also set up `.pgpass` and leave the password empty.

In [ ]:
library(glue)
# Load your local wrds package (or install it first)
devtools::load_all("..")  # if running from research/; otherwise use library(wrds)

wrds <- connect_wharton()

### Set companies and timeframe
We look up CIQ `companyid` for **Mastercard**, **Visa**, and **American Express** from `ciq.ciqcompany`, then filter transcripts by year.

In [ ]:
library(dplyr)
library(DBI)

# Company names to match (CIQ company name column)
company_names <- c("Mastercard", "Visa", "American Express")

# Look up companyid from ciq.ciqcompany (column usually 'companyname'; if query fails, check WRDS table)
name_condition <- paste(
  sprintf("LOWER(companyname) LIKE '%%%s%%'", tolower(company_names)),
  collapse = " OR "
)
lookup_sql <- sprintf(
  "SELECT companyid, companyname FROM ciq.ciqcompany WHERE %s",
  name_condition
)
company_lookup <- dbGetQuery(wrds, lookup_sql)
print(company_lookup)

companyIdValues <- company_lookup$companyid
idValuesAsString <- toString(companyIdValues)

begYear <- 2015
endYear <- 2017

### Run the transcript query
Retrieves transcript component text and metadata from:
- `ciq.wrds_transcript_detail` – transcript metadata
- `ciq.wrds_transcript_person` – speaker metadata
- `ciq.ciqtranscriptcomponent` – full transcript text

In [ ]:
queryString <- glue("
                SELECT a.*, b.*, c.componenttext
                FROM (SELECT * 
                      FROM ciq.wrds_transcript_detail
                      WHERE companyid in ({idValuesAsString}) 
                        AND date_part('year',mostimportantdateutc)>={begYear} 
                        AND date_part('year',mostimportantdateutc)<={endYear}
                     ) AS a
                JOIN ciq.wrds_transcript_person AS b
                  ON a.transcriptid = b.transcriptid
                JOIN ciq.ciqtranscriptcomponent AS c
                  ON b.transcriptcomponentid = c.transcriptcomponentid
                ORDER BY a.transcriptid, b.componentorder;
                ")

res <- dbSendQuery(wrds, queryString)

### Fetch results

In [ ]:
data <- dbFetch(res, n = -1)
dbClearResult(res)
head(data)